In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
zip_path = "/content/drive/MyDrive/celeba/img_align_celeba.zip"
extract_path = "/content/drive/MyDrive/celeba/"

if os.path.exists(zip_path):
    !unzip -q "{zip_path}" -d "{extract_path}"
    print("ZIP ochildi.")
else:
    print("ZIP topilmadi, papkani ishlatamiz.")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd

In [ ]:
class CelebADataset(Dataset):
    def __init__(self, root_dir, attr_path, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        
        # Attributes faylini o'qish
        df = pd.read_csv(attr_path, sep=r"\s+", skiprows=1)
        self.img_names = df.index.tolist()
        self.attributes = (df.values + 1) // 2  # -1 → 0, 1 → 1

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        img_path = os.path.join(self.root_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        label = torch.tensor(self.attributes[idx], dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [ ]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

In [ ]:
dataset = CelebADataset(
    root_dir="/content/drive/MyDrive/celeba/img_align_celeba",
    attr_path="/content/drive/MyDrive/celeba/list_attr_celeba.txt",
    transform=transform
)

dataloader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=2)


In [ ]:
images, labels = next(iter(dataloader))
print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

In [ ]:
import torch.nn as nn
import torch.optim as optim

class SimpleCNN(nn.Module):
    def __init__(self, num_labels=40):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(128*16*16, 512),
            nn.ReLU(),
            nn.Linear(512, num_labels),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

model = SimpleCNN()


In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0002)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

for epoch in range(5):
    running_loss = 0.0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(dataloader):.4f}")